# GeoParquet STAC Metadata

In [14]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import box, mapping
import pystac
from pystac.extensions.projection import ProjectionExtension
import datetime

In [15]:
data_url = 'path_to_object_store'

local_path = '../../downloaded_data/WAIS_Jan_2017_Polygons.parquet'

In [16]:
gdf = gpd.read_parquet(local_path)
gdf

,POLY_AREA,CENTROID_X,CENTROID_Y,Area,Shape_Leng,Shape_Area,Bedrock,Coast,GL_ON_IS,Location,...,PolsbyPopp,Fractal,Reock,Schwartzbe,LW_Ratio,Feature_Cl,Elevation,Slope,Speed,geometry
0,2025.000000,-1.119980e+06,-1.183803e+06,2068.473332,210.000000,2025.000000,492.707973,11674.572770,0.000000,Grounded Ice,...,0.577027,1.423821,0.572958,0.759623,0.750000,Lake,340,3.70869,30.86820,"POLYGON ((-1119967.5 -1183777.5, -1119967.5 -1..."
1,450.000000,-1.120432e+06,-1.184100e+06,459.648491,90.000000,450.000000,53.070199,11304.251392,0.000000,Grounded Ice,...,0.698132,1.357668,0.509296,0.835543,0.500000,Lake,340,3.70869,7.49253,"POLYGON ((-1120417.5 -1184107.5, -1120447.5 -1..."
2,258750.052500,-1.120316e+06,-1.184170e+06,264298.257963,9599.995000,258750.052500,0.000000,10604.111801,0.000000,Grounded Ice,...,0.035282,1.359245,0.152606,0.187834,0.608549,Lake,322,3.96638,12.99670,"POLYGON ((-1120042.5 -1184167.5, -1120042.5 -1..."
3,6300.000000,-1.119911e+06,-1.183954e+06,6435.242141,600.000000,6300.000000,538.188596,11491.266718,0.000000,Grounded Ice,...,0.219911,1.367579,0.163535,0.468947,0.260870,Lake,340,3.70869,45.77560,"MULTIPOLYGON (((-1119937.5 -1183867.5, -111996..."
4,450.000000,-1.119870e+06,-1.184062e+06,459.656359,90.000000,450.000000,629.150593,11454.003010,0.000000,Grounded Ice,...,0.698132,1.357668,0.509296,0.835543,0.500000,Lake,340,3.70869,45.77560,"POLYGON ((-1119862.5 -1184077.5, -1119877.5 -1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10473,384.714610,3.831272e+05,-1.027773e+06,400.316435,98.070847,384.714610,1286.411989,230918.955411,0.000000,Grounded Ice,...,0.502653,1.298060,0.299584,0.708981,0.249999,Lake,256,3.05627,72.03100,"POLYGON ((383147.295 -1027770.242, 383110.67 -..."
10474,480.895610,3.832011e+05,-1.027738e+06,500.398088,98.071048,480.895610,1248.401865,230963.424305,0.000000,Grounded Ice,...,0.628317,1.346721,0.489706,0.792665,0.666667,Lake,256,3.05627,72.03100,"POLYGON ((383211.389 -1027745.65, 383193.076 -..."
10475,481.319719,3.889432e+05,-1.058720e+06,500.396671,117.737201,481.319719,551.885201,200876.893559,209.012714,Ice Shelf,...,0.436331,1.295290,0.636616,0.660554,1.000001,Lake,87,5.09359,6.52973,"POLYGON ((388942.55 -1058704.991, 388946.064 -..."
10476,170926.591434,3.887148e+05,-1.058932e+06,177700.707329,5563.612810,170926.591434,537.851397,200310.343205,177.927304,Ice Shelf,...,0.069391,1.397146,0.579044,0.263422,0.914683,Lake,87,5.09359,6.95199,"POLYGON ((388660.338 -1059202.06, 388669.825 -..."


## Create The STAC Item And Projection Metadata


In [17]:
minx, miny, maxx, maxy = gdf.total_bounds
start, end = (datetime.datetime(2017, 1, 1, 0, 0, tzinfo=datetime.timezone.utc),
 datetime.datetime(2017, 12, 31, 0, 0, tzinfo=datetime.timezone.utc))

In [18]:
item = pystac.Item(
    id="supraglacial-lakes-boundries-only",
    geometry=mapping(box(minx, miny, maxx, maxy)),
    bbox=[minx, miny, maxx, maxy],
    datetime=start,
    properties={'title': 'Supraglacial lakes boundaries'},
    
)

proj = ProjectionExtension.ext(item, add_if_missing=True)
proj.code = gdf.crs.to_string()          # e.g. "EPSG:3031"
proj.wkt2 = gdf.crs.to_wkt()
proj.bbox = [minx, miny, maxx, maxy]    # native CRS bbox
proj.geometry = mapping(box(minx, miny, maxx, maxy))

item.extra_fields['title'] = 'Subglacial lakes boundaries data'
item

<Item id=supraglacial-lakes-boundries-only>

## Create The Parquet Asset

The asset records where the Parquet file will live and identifies it as a data asset. The href remains the original placeholder until the object-store path is supplied.


In [19]:
asset = pystac.Asset(href=data_url, media_type="application/x-parquet", roles=["data"])

## Describe The Parquet Columns


In [20]:
asset.extra_fields["parquet:columns"] = [
    {
        "name": "POLY_AREA",
        "type": "float",
        "long_name": "Polygon area",
        "description": (
            "Area of the supraglacial lake polygon as provided in the source vector dataset. "
            "This may duplicate or closely correspond to Shape_Area or Area, depending on the "
            "source processing workflow."
        ),
        "unit": "m2",
    },
    {
        "name": "CENTROID_X",
        "type": "float",
        "long_name": "Lake centroid longitude",
        "description": (
            "X coordinate of the supraglacial lake polygon centroid. For Antarctic geographic "
            "datasets this is typically longitude in decimal degrees, unless the source layer "
            "uses a projected coordinate reference system."
        ),
        "unit": "degree",
    },
    {
        "name": "CENTROID_Y",
        "type": "float",
        "long_name": "Lake centroid latitude",
        "description": (
            "Y coordinate of the supraglacial lake polygon centroid. For Antarctic geographic "
            "datasets this is typically latitude in decimal degrees, unless the source layer "
            "uses a projected coordinate reference system."
        ),
        "unit": "degree",
    },
    {
        "name": "Area",
        "type": "float",
        "long_name": "Supraglacial lake area",
        "description": (
            "Surface area of the mapped supraglacial lake feature. Used to describe lake size "
            "and to support derived metrics such as volume estimates and area-perimeter ratios."
        ),
        "unit": "m2",
    },
    {
        "name": "Shape_Leng",
        "type": "float",
        "long_name": "Lake polygon perimeter",
        "description": (
            "Perimeter or boundary length of the supraglacial lake polygon geometry."
        ),
        "unit": "m",
    },
    {
        "name": "Shape_Area",
        "type": "float",
        "long_name": "Lake polygon shape area",
        "description": (
            "Area of the lake polygon geometry calculated by the source GIS software. This may "
            "duplicate or closely correspond to POLY_AREA or Area."
        ),
        "unit": "m2",
    },
    {
        "name": "Bedrock",
        "type": "float",
        "long_name": "Distance to exposed bedrock",
        "description": (
            "Distance from the supraglacial lake feature to the nearest mapped exposed bedrock "
            "or rock outcrop, where available."
        ),
        "unit": "m",
    },
    {
        "name": "Coast",
        "type": "float",
        "long_name": "Distance to coast",
        "description": (
            "Distance from the supraglacial lake feature to the Antarctic coastline or ice-shelf "
            "front used in the source analysis."
        ),
        "unit": "m",
    },
    {
        "name": "GL_ON_IS",
        "type": "boolean",
        "long_name": "Grounding-line-on-ice-shelf flag",
        "description": (
            "Flag indicating whether the supraglacial lake is located on an ice shelf or in a "
            "grounding-line-related ice-shelf setting, according to the source classification."
        ),
        "unit": None,
    },
    {
        "name": "Location",
        "type": "string",
        "long_name": "Lake location class",
        "description": (
            "Textual location category or regional setting assigned to the supraglacial lake, "
            "such as ice sheet, ice shelf, grounding-zone, coastal, or named regional grouping, "
            "depending on the source classification."
        ),
        "unit": None,
    },
    {
        "name": "GL",
        "type": "float",
        "long_name": "Distance to grounding line",
        "description": (
            "Distance from the supraglacial lake feature to the Antarctic grounding line used "
            "in the source analysis."
        ),
        "unit": "m",
    },
    {
        "name": "Volume",
        "type": "float",
        "long_name": "Estimated lake volume",
        "description": (
            "Estimated water volume of the supraglacial lake feature, typically derived from "
            "mapped lake area and an empirical or remotely sensed depth relationship."
        ),
        "unit": "m3",
    },
    {
        "name": "A_P_Ratio",
        "type": "float",
        "long_name": "Area-perimeter ratio",
        "description": (
            "Ratio of lake polygon area to perimeter, used as a compactness or shape metric for "
            "the mapped supraglacial lake."
        ),
        "unit": "m",
    },
    {
        "name": "REMA_Max",
        "type": "float",
        "long_name": "Maximum REMA elevation",
        "description": (
            "Maximum surface elevation from the Reference Elevation Model of Antarctica within "
            "or associated with the supraglacial lake polygon."
        ),
        "unit": "m",
    },
    {
        "name": "REMA_Min",
        "type": "float",
        "long_name": "Minimum REMA elevation",
        "description": (
            "Minimum surface elevation from the Reference Elevation Model of Antarctica within "
            "or associated with the supraglacial lake polygon."
        ),
        "unit": "m",
    },
    {
        "name": "PolsbyPopp",
        "type": "float",
        "long_name": "Polsby-Popper compactness",
        "description": (
            "Shape compactness metric for the lake polygon based on the ratio of polygon area "
            "to the area of a circle with the same perimeter. Values closer to 1 indicate a more "
            "compact, circular shape."
        ),
        "unit": "1",
    },
    {
        "name": "Fractal",
        "type": "float",
        "long_name": "Fractal dimension shape index",
        "description": (
            "Shape-complexity metric describing the irregularity of the supraglacial lake polygon "
            "boundary. Higher values generally indicate a more complex or convoluted outline."
        ),
        "unit": "1",
    },
    {
        "name": "Reock",
        "type": "float",
        "long_name": "Reock compactness",
        "description": (
            "Shape compactness metric comparing the polygon area with the area of its minimum "
            "bounding circle. Values closer to 1 indicate a more compact shape."
        ),
        "unit": "1",
    },
    {
        "name": "Schwartzbe",
        "type": "float",
        "long_name": "Schwartzberg compactness",
        "description": (
            "Shape compactness metric comparing the polygon perimeter with the circumference of "
            "a circle of equal area. Values closer to 1 generally indicate a more compact, "
            "circular polygon."
        ),
        "unit": "1",
    },
    {
        "name": "LW_Ratio",
        "type": "float",
        "long_name": "Length-width ratio",
        "description": (
            "Ratio of lake length to lake width, used to describe elongation of the supraglacial "
            "lake polygon."
        ),
        "unit": "1",
    },
    {
        "name": "Feature_Cl",
        "type": "string",
        "long_name": "Feature class",
        "description": (
            "Classification assigned to the mapped feature in the source dataset. For this asset, "
            "features represent supraglacial lakes, but this field may preserve source-level "
            "feature class labels or quality classes."
        ),
        "unit": None,
    },
    {
        "name": "Elevation",
        "type": "float",
        "long_name": "Lake surface elevation",
        "description": (
            "Representative surface elevation associated with the supraglacial lake feature, "
            "likely derived from REMA or another Antarctic digital elevation model."
        ),
        "unit": "m",
    },
    {
        "name": "Slope",
        "type": "float",
        "long_name": "Surface slope",
        "description": (
            "Representative ice-surface slope associated with the supraglacial lake feature."
        ),
        "unit": "degree",
    },
    {
        "name": "Speed",
        "type": "float",
        "long_name": "Ice surface velocity",
        "description": (
            "Representative ice-flow speed at or near the supraglacial lake feature, derived from "
            "an Antarctic ice-velocity product used in the source analysis."
        ),
        "unit": "m yr-1",
    },
    {
        "name": "geometry",
        "type": "geometry",
        "long_name": "Supraglacial lake polygon geometry",
        "description": (
            "Polygon geometry representing the mapped supraglacial lake surface extent. Stored as "
            "the GeoParquet geometry column."
        ),
        "unit": None,
    },
]

In [21]:
item.add_asset('data', asset)

In [22]:
item.validate()
item

<Item id=supraglacial-lakes-boundries-only>